In [0]:
%sql
show tables from snow_pro_dev.snowpro_l0

In [0]:
%sql
with cleaned_income as 
(select *, 
       cast(regexp_replace(ANNUALINCOME, '[\\$,]', '') as int) as ANNUALINCOME_INT,
       cast(regexp_replace(ANNUALINCOME, '[$,]' , '') as int ) as Final_income   
from snow_pro_dev.snowpro_l0.adventureworks_customers)

, rank 
(select CUSTOMERKEY, annualincome_int, concat_ws('_', FIRSTNAME, LASTNAME) as name,
row_number() over (partition by CUSTOMERKEY ORDER BY annualincome_int) as rank
from cleaned_income)

  select * from rank
  where rank in (2,3) 


In [0]:
%sql
create table snow_pro_dev.snowpro_l1.window_key1 as  
with ord_key 
(
  select ORDERNUMBER, TERRITORYKEY from snow_pro_dev.snowpro_l0.advent_works
  )

, row_key 
(
  select ordernumber, territorykey,
row_number() over (partition by ORDERNUMBER ORDER BY TERRITORYKEY) as rowkey
from ord_key
)

, dense_key
(
  select ordernumber, territorykey,
dense_rank() over (partition by ORDERNUMBER ORDER BY TERRITORYKEY) as densekey
from ord_key
)

, rank_key
(
  select ordernumber, territorykey, 
rank() over (partition by ordernumber order by territorykey) as rankey
from ord_key
)

-- select ordernumber, territorykey, 
-- percent_rank() over (partition by ordernumber order by territorykey) as per_key
-- from ord_key

select ord_key.ordernumber, ord_key.territorykey, dense_key.densekey, row_key.rowkey, rank_key.rankey from ord_key
left join row_key on ord_key.ordernumber = row_key.ordernumber 
left join dense_key on row_key.ordernumber = dense_key.ordernumber
left join rank_key on dense_key.ordernumber = rank_key.ordernumber

In [0]:
%sql
 create table snow_pro_dev.snowpro_l1.functions_key1
 with id 
 (
  select * from esat_snow_dev.sales_l0.sales
 )

 , dense
 (select *,
 dense_rank() over(partition by product_id order by profit) as rank 
 from id)

 , toprank 
 (select * from dense
 where rank in (2,3)
 order by product_id)

 , link
 (select a.customer_id, a.product_id, b.category, b.brand, c.gender, c.age, a.rank from toprank as a
 left join esat_snow_dev.sales_l0.products as b on a.product_id = b.product_id
 left join esat_snow_dev.sales_l0.customers as c on a.customer_id = c.customer_id)

 , gender 
 (select brand, category, gender, count(gender) as count_gender from link
 group by brand, category, gender
 order by count_gender desc)

 select *,
 rank() over (partition by gender order by count_gender desc) as rank1 
 from gender

In [0]:
%sql
select * from esat_snow_dev.sales_l0.products

In [0]:
%sql
select * from esat_snow_dev.sales_l0.customers

In [0]:
%sql    
-- create table product_year as

with pro_yr
(select 
year(ORDERDATE) as year,
month(ORDERDATE) as month,
-- date(orderdate) as date,
sum (ORDERQUANTITY) as total_quantity
 from snow_pro_dev.snowpro_l0.adventureworks_sales_2016
group by year(ORDERDATE),
        month(ORDERDATE)
order by year, month)

select *,
total_quantity - lag (total_quantity) over (order by year, month) as diff,
total_quantity - lead (total_quantity) over (order by year, month) as diff1,
sum (total_quantity) over (order by year, month) as diff2
from pro_yr


In [0]:
%sql
select PRODUCTCATEGORYKEY, CATEGORYNAME from snow_pro_dev.snowpro_l0.adventureworks_product_categories
group by PRODUCTCATEGORYKEY, CATEGORYNAME 


In [0]:
%sql
select * from esat_snow_dev.sales_l0.products
where category NOT IN 
(select category FROM snow_pro_dev.snowpro_l1.functions_key1 
where functions_key1.category NOT in ('Truffle', 'Milk', 'Praline'))

In [0]:
%sql
select * FROM snow_pro_dev.snowpro_l1.functions_key1 
-- where functions_key1.category not in ('Truffle', 'Milk', 'Praline')

In [0]:
%sql
create table snow_pro_dev.snowpro_l1.brand_code1 

with code
(
    SELECT DISTINCT brand,
CASE
WHEN brand = 'Mars' then 'B001'
WHEN brand = 'Lindt' then 'B002'
WHEN brand = 'Hershey' then 'B003'
WHEN brand = 'Godiva' then 'B004'
WHEN brand = 'Cadbury' then 'B005'
WHEN brand = 'Ferrero' then 'B006'
end as Code
FROM esat_snow_dev.sales_l0.products 
order by code
)

select a.code, b.* from code as a
left join snow_pro_dev.snowpro_l1.functions_key1 as b on a.brand = b.brand
order by code